In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

In [7]:
# reusable function for generating LLM
from langchain_google_genai import ChatGoogleGenerativeAI

def get_llm():
    return ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        vertexai=True
    )

## Define the State

In [1]:
from typing import List, Dict, Any, TypedDict, Optional

In [2]:
class AssistantInfo(TypedDict):
    assistant_type: str
    assistant_instructions: str
    user_question: str


class SearchQuery(TypedDict):
    search_query: str
    user_question: str

class SearchResult(TypedDict):
    result_url: str
    search_query: str
    user_question: str
    is_fallback: Optional[bool]

class SearchSummary(TypedDict):
    summary: str
    result_url: str
    user_question: str
    is_fallback: Optional[bool]

class ResearchReport(TypedDict):
    report: str

Overall state for the Agentic Graph

In [3]:
class ResearchState(TypedDict):
    user_question: str
    assistant_info: Optional[AssistantInfo]
    search_queries: Optional[List[SearchQuery]]
    search_results: Optional[List[SearchResult]]
    search_summaries: Optional[List[SearchSummary]]
    research_summary: Optional[str]
    final_report: Optional[str]
    used_fallback_search: Optional[bool]
    relevance_evaluation: Optional[Dict[str, Any]]
    should_regenerate_queries: Optional[bool]
    iteration_count: Optional[int]

## Define node function to handle process
For each node, we should define
- <u>Input state</u>: The data each node requires to function
- <u>Processing</u>: The task that each node performs
- <u>State Update</u>: The information each  node returns to update the overall state
We only put one node function on this notebook for demonstration purpose. The other node functions are in package.

In [4]:
from langchain_core.prompts import PromptTemplate

In [5]:
# the prompt for the node function
# RESEARCH REPORT
# Research Report prompts adapted from https://github.com/assafelovic/gpt-researcher/blob/master/gpt_researcher/master/prompts.py
RESEARCH_REPORT_INSTRUCTIONS = """
You are an AI critical thinker research assistant. Your sole purpose is to write well written, critically acclaimed, objective and structured reports on given text.

Information: 
--------
{research_summary}
--------

Using the above information, answer the following question or topic: "{user_question}" in a detailed report -- \
The report should focus on the answer to the question, should be well structured, informative, \
in depth, with facts and numbers if available and a minimum of 1,200 words.

You should strive to write the report as long as you can using all relevant and necessary information provided.
You must write the report with markdown syntax.
You MUST determine your own concrete and valid opinion based on the given information. Do NOT deter to general and meaningless conclusions.
Write all used source urls at the end of the report, and make sure to not add duplicated sources, but only one reference for each.
You must write the report in apa format.
Please do your best, this is very important to my career.""" 

RESEARCH_REPORT_PROMPT_TEMPLATE = PromptTemplate.from_template(
    template=RESEARCH_REPORT_INSTRUCTIONS
)

In [6]:
# node function for writting  report for research result

def write_research_report(state:Dict[str, Any]) -> Dict[str, Any]:
    """
    Write a research report based on the summarized search results.
    """
    research_summary = state["research_summary"]
    user_question = state["user_question"]
    
    # Format the prompt
    prompt = RESEARCH_REPORT_PROMPT_TEMPLATE.format(
        research_summary=research_summary,
        user_question=user_question
    )
    
    # Get the LLM response
    llm = get_llm()
    response = llm.invoke(prompt)
    report = response.content
    
    # Return the updated state
    return {"final_report": report}

In [7]:
from node_functions import select_assistant, generate_search_queries, perform_web_searches, summarize_search_results, evaluate_search_relevance, write_research_report 

### define a conditional edge and node function

In [8]:
def route_based_on_relevance(state):
    iteration_count = state.get("iteration_count", 0) + 1
    state["iteration_count"] = iteration_count
    if iteration_count >= 3:
        return "write_research_report"
    if state.get("should_regenerate_queries", False):
        return "generate_search_queries"
    return "write_research_report"

## Define the graph structure

In [9]:
from langgraph.graph import StateGraph, END

In [10]:
# Define the graph
graph = StateGraph(ResearchState)

In [11]:
# Add nodes to the graph
graph.add_node("select_assistant", select_assistant)
graph.add_node("generate_search_queries", generate_search_queries)
graph.add_node("perform_web_searches", perform_web_searches)
graph.add_node("summarize_search_results", summarize_search_results)
graph.add_node("evaluate_search_relevance", evaluate_search_relevance)
graph.add_node("write_research_report", write_research_report)

### conditional node 
#graph.add_node("route_based_on_relevance", route_based_on_relevance)

In [12]:
# Define the flow of the graph by adding edges
graph.add_edge("select_assistant", "generate_search_queries")
graph.add_edge("generate_search_queries", "perform_web_searches")
graph.add_edge("perform_web_searches", "summarize_search_results")
graph.add_edge("summarize_search_results", "evaluate_search_relevance")

In [13]:
# Add conditional routing based on relevance evaluation
graph.add_conditional_edges(
    "evaluate_search_relevance",
    route_based_on_relevance,
    {
        "generate_search_queries": "generate_search_queries",
        "write_research_report": "write_research_report"
    }
)

In [14]:
# set the end point
graph.add_edge("write_research_report", END)

In [15]:
# Set the entry point
graph.set_entry_point("select_assistant")

## Compile and test

In [16]:
agent = graph.compile()

In [17]:
initial_state = {
    "user_question": " What can you tell me about Astorga's roman spas?",
    "assistant_info": None,
    "search_queries": None,
    "search_results": None,
    "search_summaries": None,
    "research_summary": None,
    "final_report": None,
    "used_fallback_search": False,
    "relevance_evaluation": None,
    "should_regenerate_queries": None,
    "iteration_count": 0
}

result = agent.invoke(initial_state)

Generating initial search queries...
Generated 3 search queries
  Query 1: Astorga Roman spas archaeological sites
  Query 2: Roman baths of Astorga history and function
  Query 3: Excavations and discoveries Roman spas Astorga
Performing web searches for 3 queries...
Searching for: Astorga Roman spas archaeological sites
Fallback search was used for query: Astorga Roman spas archaeological sites
Found 3 results for query: Astorga Roman spas archaeological sites
Searching for: Roman baths of Astorga history and function
Found 3 results for query: Roman baths of Astorga history and function
Searching for: Excavations and discoveries Roman spas Astorga
Fallback search was used for query: Excavations and discoveries Roman spas Astorga
Found 3 results for query: Excavations and discoveries Roman spas Astorga
Summarizing 9 search results...
Scraping content from: https://en.wikipedia.org/wiki/List_of_Roman_sites_in_Spain
Skipping https://en.wikipedia.org/wiki/List_of_Roman_sites_in_Spain du

In [18]:
from rich import print as pprint

In [19]:
final_report = result["final_report"]
pprint(final_report)

## The Enduring Legacy of Roman Hydration: An Examination of Astorga's Roman Spas

Astorga, a city steeped in Roman history, offers a compelling glimpse into the sophisticated urban infrastructure 
and social practices of the Roman Empire, particularly through its well-preserved thermal bath complexes. While the
provided information on Astorga's Roman spas is somewhat fragmented, a synthesis of the available details allows 
for a comprehensive understanding of their significance, location, and continued presence within the archaeological
landscape. The available sources indicate the existence of at least two distinct bathing complexes: the "Termas 
Mayores" (Greater Baths) and "termas menores" (smaller baths), both integral components of the Roman settlement of 
Asturica Augusta. These structures not only served a vital hygienic function but also represented centers of social
interaction and civic pride, reflecting the Roman emphasis on public amenities and the integration of these 
facilities into the daily lives of its citizens.

**The Roman Presence in Astorga: Asturica Augusta**

Before delving into the specifics of the spas, it is crucial to contextualize them within the broader Roman history
of Astorga. The city, known in Roman times as Asturica Augusta, was a significant administrative and military 
center in the northwestern Hispania. Its strategic location, particularly its connection to important routes like 
the Vía de la Plata (Silver Route), which linked it to Seville and passed by the Roman trading center of Cáparra, 
underscored its economic and logistical importance. The Vía de la Plata, a historic Roman road, served as a vital 
artery for trade and communication, and Astorga's position along this route would have facilitated the movement of 
goods, people, and military personnel, thereby contributing to its growth and prosperity.

The Roman presence in Astorga is further evidenced by the extensive archaeological findings that can be explored 
along the "Roman Route." This route encompasses a variety of Roman ruins, including sections of the basilica, 
thermal baths, mansions, and parts of the sewerage system. This comprehensive network of preserved structures 
highlights the advanced urban planning and engineering capabilities of the Romans, who meticulously designed their 
settlements to cater to the needs of their inhabitants. The inclusion of thermal baths as a key component of this 
Roman Route underscores their paramount importance in Roman daily life.

**The Termas Mayores: Centrality and Scale**

The "Termas Mayores," or Greater Baths, are identified as being located in the central part of the city, situated 
between the two main streets of Roman Asturica Augusta. This prime location suggests a deliberate placement to 
ensure accessibility for a broad segment of the population. Public baths in Roman cities were not merely places for
hygiene; they were vibrant social hubs where citizens engaged in conversation, conducted business, exercised, and 
relaxed. The central positioning of the Termas Mayores would have amplified their role as a focal point of urban 
life, fostering a sense of community and shared experience.

While specific dimensions or capacities of the Termas Mayores are not provided in the given text, the designation 
"Greater Baths" implies a substantial scale of construction and a comprehensive range of facilities. Roman thermae 
typically included various rooms such as *frigidarium* (cold bath), *tepidarium* (warm bath), and *caldarium* (hot 
bath), often accompanied by *palaestrae* (exercise grounds), changing rooms (*apodyteria*), and sometimes even 
libraries or gardens. The sheer size and complexity of such establishments required significant engineering and 
resources, reflecting the wealth and organizational capacity of Asturica Augusta. The fact that these baths are 
referred to as "Mayores" suggests that they were the primary and most elaborate bathing complex within the city, 
ca